# セットアップ

`data/csv` 配下のCSVファイルから、指定したカタログ・スキーマにテーブルを作成します。

In [0]:
dbutils.widgets.text("catalog", "", "カタログ名")
dbutils.widgets.text("schema", "", "スキーマ名")

In [0]:
import os
import pandas as pd
from pathlib import Path
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, DoubleType, BooleanType, DateType, TimestampType,
)

# パラメータの取得
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema")

assert catalog, "カタログ名を指定してください"
assert schema_name, "スキーマ名を指定してください"

# ── テーブルごとのスキーマ定義 ──────────────────────────────────
TABLE_SCHEMAS = {
    "agents": StructType([
        StructField("agent_id", StringType(), False),
        StructField("agent_name", StringType(), True),
        StructField("team", StringType(), True),
        StructField("tier", StringType(), True),
        StructField("location", StringType(), True),
        StructField("hire_date", DateType(), True),
    ]),
    "customers": StructType([
        StructField("customer_id", StringType(), False),
        StructField("customer_name", StringType(), True),
        StructField("segment", StringType(), True),
        StructField("region", StringType(), True),
        StructField("industry", StringType(), True),
        StructField("plan_tier", StringType(), True),
        StructField("mrr_jpy", LongType(), True),
        StructField("contract_start_date", DateType(), True),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), False),
        StructField("product_name", StringType(), True),
        StructField("module", StringType(), True),
        StructField("release_version", StringType(), True),
    ]),
    "sla_policies": StructType([
        StructField("priority", StringType(), False),
        StructField("sla_target_hours", IntegerType(), True),
        StructField("business_hours_only", BooleanType(), True),
    ]),
    "support_tickets": StructType([
        StructField("ticket_id", StringType(), False),
        StructField("customer_id", StringType(), True),
        StructField("product_id", StringType(), True),
        StructField("agent_id", StringType(), True),
        StructField("category", StringType(), True),
        StructField("priority", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("open_date", TimestampType(), True),
        StructField("close_date", TimestampType(), True),
        StructField("resolution_time_hours", DoubleType(), True),
        StructField("reopened_count", IntegerType(), True),
        StructField("csat_score", IntegerType(), True),
        StructField("status", StringType(), True),
    ]),
}

# ── 相対パスでCSVディレクトリを特定 ────────────────────────────
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
csv_dir = f"/Workspace{notebook_dir}/data/csv"

csv_files = sorted([f for f in os.listdir(csv_dir) if f.endswith(".csv")])
print(f"対象CSVファイル: {csv_files}")

# ── 各CSVファイルからテーブルを作成 ────────────────────────────
for csv_file in csv_files:
    table_name = Path(csv_file).stem
    full_table_name = f"{catalog}.{schema_name}.{table_name}"
    csv_path = f"{csv_dir}/{csv_file}"

    table_schema = TABLE_SCHEMAS.get(table_name)
    if table_schema is None:
        print(f"⚠️ スキーマ未定義のためスキップ: {table_name}")
        continue

    # CSV読み込み & 日付/タイムスタンプ列を変換
    pdf = pd.read_csv(csv_path)
    for field in table_schema:
        if isinstance(field.dataType, DateType):
            pdf[field.name] = pd.to_datetime(pdf[field.name]).dt.date
        elif isinstance(field.dataType, TimestampType):
            pdf[field.name] = pd.to_datetime(pdf[field.name])

    df = spark.createDataFrame(pdf, schema=table_schema)
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
    print(f"✅ テーブル作成完了: {full_table_name}")

print(f"\n🎉 全 {len(csv_files)} テーブルの作成が完了しました")

In [0]:
import os
import shutil

# パラメータの取得
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema")

# Volume の作成
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_name}.files")
print(f"✅ Volume 作成完了: {catalog}.{schema_name}.files")

# 相対パスで非構造化データディレクトリを特定
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(notebook_path)
src_dir = f"/Workspace{notebook_dir}/data/unstructured_data"
dst_dir = f"/Volumes/{catalog}/{schema_name}/files"

# ファイルをコピー
files = sorted([f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))])
for file_name in files:
    shutil.copy2(os.path.join(src_dir, file_name), os.path.join(dst_dir, file_name))
    print(f"✅ コピー完了: {file_name}")

print(f"\n🎉 全 {len(files)} ファイルのコピーが完了しました")
print(f"   Volume パス: {dst_dir}")